# kor-minish v2 fine-tune (Colab T4)

**흐름**: KLUE-STS + KLUE-NLI + PAWS-X(ko) → BAAI/bge-m3 fine-tune (CosineSimilarityLoss) → model2vec 재-distill → 평가 → zip 다운로드

**데이터 (HF에서 자동 다운로드)**:
- KLUE-STS (12K, 0~5 점수)
- KLUE-NLI (25K, entailment/neutral/contradiction)
- PAWS-X 한국어 (49K, paraphrase 쌍) ← 챗봇 매칭에 핵심
- 총 ~86K 쌍

**예상**: T4 GPU 약 60~90분. Runtime → Change runtime type → **T4 GPU**

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q 'sentence-transformers>=3.0' 'model2vec[distill]>=0.8' 'datasets>=2.19' 'huggingface_hub>=0.24'

In [ ]:
from datasets import load_dataset
from sentence_transformers import InputExample
import random

NLI_SCORE = {0: 1.0, 1: 0.5, 2: 0.0}
train_examples = []
stats = {}

def add(name, count):
    stats[name] = count
    print(f'  + {name}: {count:,} pairs')

# 1) KLUE-STS
try:
    ds = load_dataset('klue', 'sts', split='train')
    n = 0
    for row in ds:
        score = row['labels']['label'] / 5.0
        train_examples.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=score))
        n += 1
    add('klue-sts', n)
except Exception as e:
    print(f'  - klue-sts failed: {e}')

# 2) KLUE-NLI
try:
    ds = load_dataset('klue', 'nli', split='train')
    n = 0
    for row in ds:
        score = NLI_SCORE.get(row['label'], 0.5)
        train_examples.append(InputExample(texts=[row['premise'], row['hypothesis']], label=score))
        n += 1
    add('klue-nli', n)
except Exception as e:
    print(f'  - klue-nli failed: {e}')

# 3) PAWS-X 한국어 (paraphrase) — 챗봇 매칭에 핵심
try:
    ds = load_dataset('paws-x', 'ko', split='train')
    n = 0
    for row in ds:
        score = 1.0 if row['label'] == 1 else 0.0
        train_examples.append(InputExample(texts=[row['sentence1'], row['sentence2']], label=score))
        n += 1
    add('paws-x-ko', n)
except Exception as e:
    print(f'  - paws-x failed: {e}')

random.shuffle(train_examples)
print(f'\n총 학습 쌍: {len(train_examples):,}')
for ex in train_examples[:3]:
    print(f'  {ex.label:.2f}  | {ex.texts[0][:40]:40s} | {ex.texts[1][:40]}')

In [ ]:
# bge-m3 fine-tune
from sentence_transformers import SentenceTransformer, losses
from torch.utils.data import DataLoader

BASE = 'BAAI/bge-m3'
EPOCHS = 1
BATCH = 16  # T4 16GB. OOM 나면 8로

model = SentenceTransformer(BASE, device='cuda')
loader = DataLoader(train_examples, shuffle=True, batch_size=BATCH)
loss = losses.CosineSimilarityLoss(model)
warmup_steps = int(len(loader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(loader, loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    output_path='bge-m3-ko-finetuned',
    show_progress_bar=True,
)
print('fine-tune 완료')

In [ ]:
# Fine-tuned 모델 → model2vec 재-distill (한국어 vocab 같이 주입)
import requests
from model2vec.distill import distill

VOCAB_URL = 'https://tmpfiles.org/dl/w4wSwgrVgrJp/vocab_ko.txt'
try:
    r = requests.get(VOCAB_URL, headers={'User-Agent': 'Mozilla/5.0'})
    r.raise_for_status()
    open('vocab_ko.txt', 'wb').write(r.content)
    vocab = [l.strip() for l in open('vocab_ko.txt', encoding='utf-8') if l.strip()]
    print(f'vocab loaded: {len(vocab):,}')
except Exception as e:
    print(f'vocab download failed ({e}); skipping vocabulary injection')
    vocab = None

m2v = distill(
    model_name='bge-m3-ko-finetuned',
    vocabulary=vocab,
    pca_dims=256,
    device='cuda',
)
m2v.save_pretrained('kor-minish-bge-m3-ko-v2')
print('saved -> kor-minish-bge-m3-ko-v2')

In [ ]:
# v1 vs v2 비교 평가
import numpy as np
from model2vec import StaticModel

v2 = StaticModel.from_pretrained('kor-minish-bge-m3-ko-v2')
v1 = StaticModel.from_pretrained('hysnnnn/kor-minish-bge-m3-ko')

def cos(a, b): return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

SYNONYM = [
    ('월급이 인상되었다', '급여가 올랐다'),
    ('이 책은 정말 흥미롭다', '이 도서는 매우 재미있다'),
    ('서울에서 부산까지 KTX', '서울발 부산행 고속열차'),
    ('돈 돌려받고 싶어요', '환불 신청 방법'),
    ('비번 까먹었어요', '비밀번호 찾기'),
    ('아 그냥 안 살래요', '주문 취소'),
    ('할인 코드 어디에 넣어요', '쿠폰 사용법'),
]
UNRELATED = [
    ('김치찌개 레시피', '자동차 엔진 정비'),
    ('주식 시장 폭락', '강아지 산책 코스'),
]
print(f'{"문장쌍":50s}  v1     v2     diff')
print('-' * 80)
print('== 동의 (높을수록 좋음) ==')
v1_syn, v2_syn = [], []
for a, b in SYNONYM:
    s1, s2 = cos(*v1.encode([a, b])), cos(*v2.encode([a, b]))
    v1_syn.append(s1); v2_syn.append(s2)
    print(f'{a[:22]:22s} ↔ {b[:22]:22s}  {s1:+.3f}  {s2:+.3f}  {s2-s1:+.3f}')
print('== 무관 (낮을수록 좋음) ==')
v1_unr, v2_unr = [], []
for a, b in UNRELATED:
    s1, s2 = cos(*v1.encode([a, b])), cos(*v2.encode([a, b]))
    v1_unr.append(s1); v2_unr.append(s2)
    print(f'{a[:22]:22s} ↔ {b[:22]:22s}  {s1:+.3f}  {s2:+.3f}  {s2-s1:+.3f}')
print()
print(f'동의 평균:  v1={sum(v1_syn)/len(v1_syn):+.3f}  v2={sum(v2_syn)/len(v2_syn):+.3f}')
print(f'무관 평균:  v1={sum(v1_unr)/len(v1_unr):+.3f}  v2={sum(v2_unr)/len(v2_unr):+.3f}')
print(f'Margin:    v1={sum(v1_syn)/len(v1_syn) - sum(v1_unr)/len(v1_unr):+.3f}  '
      f'v2={sum(v2_syn)/len(v2_syn) - sum(v2_unr)/len(v2_unr):+.3f}')

In [ ]:
import shutil
from google.colab import files
shutil.make_archive('kor-minish-bge-m3-ko-v2', 'zip', 'kor-minish-bge-m3-ko-v2')
files.download('kor-minish-bge-m3-ko-v2.zip')